In [1]:
# ── Imports ────────────────────────────────────────────────────────────────
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from adjustText import adjust_text
from matplotlib.lines import Line2D
import warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '.')
from utils import fill_deaths, SMALL_CONSTANT, DATA_DIR, OUT_DIR


# ── Helper functions ───────────────────────────────────────────────────────
# fill_deaths imported from utils.py

def build_town_averages(panel):
    """Aggregate panel to one row per town: per-month averages then log metric."""
    n_months = panel.groupby('admin2_name')['year_month'].count().rename('n_months')
    town = (
        panel.groupby('admin2_name')
        .agg(sum_aid=('total_aid_spend', 'sum'), sum_deaths=('deaths_filled', 'sum'))
        .join(n_months)
        .reset_index()
    )
    town['avg_aid_pm']    = town['sum_aid']    / town['n_months']
    town['avg_deaths_pm'] = town['sum_deaths'] / town['n_months']
    town['log_aid']       = np.log1p(town['avg_aid_pm'])
    town['log_deaths']    = np.log1p(town['avg_deaths_pm'])
    town['metric']        = town['log_aid'] - town['log_deaths']
    return town


def categorize_towns(town):
    """Label each town as underserved / overserved / other using quartile thresholds.
    Vectorised — no row-by-row apply loop."""
    p25_metric = town['metric'].quantile(0.25)
    p75_metric = town['metric'].quantile(0.75)
    p75_aid    = town['log_aid'].quantile(0.75)

    conditions = [
        town['metric'] <= p25_metric,
        (town['metric'] >= p75_metric) & (town['log_aid'] >= p75_aid),
    ]
    choices = ['underserved', 'overserved']
    town = town.copy()
    town['category'] = np.select(conditions, choices, default='other')
    return town, p25_metric, p75_metric, p75_aid


def top5_labels(town):
    """Return the 5 most extreme underserved and 5 most extreme overserved towns."""
    under = town[town['category'] == 'underserved'].copy()
    under['score'] = under['log_deaths'].rank(pct=True) - under['log_aid'].rank(pct=True)
    top_under = under.nlargest(5, 'score')

    over = town[town['category'] == 'overserved'].copy()
    top_over = over.nlargest(5, 'log_aid')
    return top_under, top_over


# ── Load & prep ────────────────────────────────────────────────────────────
panel = pd.read_csv(DATA_DIR / 'violence_aid_merged.csv')
panel['year_month'] = pd.to_datetime(panel['year_month'], format='%Y-%m')
panel = panel.sort_values(['admin2_name', 'year_month'])
print(f'Loaded {len(panel):,} rows | {panel["admin2_name"].nunique()} towns | {panel["year_month"].nunique()} months')

panel['deaths_filled'] = panel.groupby('admin2_name')['total_deaths'].transform(fill_deaths)

# ── Aggregate to town-level ────────────────────────────────────────────────
town = build_town_averages(panel)
town, p25, p75_m, p75_a = categorize_towns(town)
print(f'\nCategory counts:\n{town["category"].value_counts().to_string()}')
print(f'\nMetric thresholds — underserved ≤ {p25:.2f} | overserved ≥ {p75_m:.2f} (metric) & ≥ {p75_a:.2f} (log aid)')

top5_under, top5_over = top5_labels(town)

# ── Plot ───────────────────────────────────────────────────────────────────
color_map = {'underserved': 'firebrick', 'overserved': 'darkorange', 'other': 'steelblue'}

fig, ax = plt.subplots(figsize=(13, 9))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

for cat, zorder, alpha, size in [('other', 3, 0.45, 55),
                                   ('overserved', 4, 0.75, 75),
                                   ('underserved', 5, 0.85, 80)]:
    mask = town['category'] == cat
    ax.scatter(town.loc[mask, 'log_deaths'], town.loc[mask, 'log_aid'],
               c=color_map[cat], alpha=alpha, edgecolors='white',
               linewidths=0.4, s=size, zorder=zorder)

# Median reference lines
ax.axvline(town['log_deaths'].median(), color='gray', linestyle='--', linewidth=1, zorder=2)
ax.axhline(town['log_aid'].median(),    color='gray', linestyle='--', linewidth=1, zorder=2)

xmin, xmax = ax.get_xlim()
ymin, ymax = ax.get_ylim()
ax.text(xmax*0.98, ymax*0.98,             'Low deaths, High aid',    ha='right', va='top',    fontsize=8.5, color='gray',      fontstyle='italic')
ax.text(xmax*0.98, ymin+(ymax-ymin)*0.02, 'High deaths, Low aid ⚠', ha='right', va='bottom', fontsize=8.5, color='firebrick', fontstyle='italic', fontweight='bold')
ax.text(xmin+(xmax-xmin)*0.02, ymax*0.98,             'Low deaths, High aid', ha='left', va='top',    fontsize=8.5, color='gray', fontstyle='italic')
ax.text(xmin+(xmax-xmin)*0.02, ymin+(ymax-ymin)*0.02, 'Low deaths, Low aid',  ha='left', va='bottom', fontsize=8.5, color='gray', fontstyle='italic')

# Labels for top-5 underserved and overserved
texts = []
for _, r in top5_under.iterrows():
    y_offset = -0.6 if r['admin2_name'] == 'Kwamouth' else 0
    texts.append(ax.text(r['log_deaths'], r['log_aid'] + y_offset, r['admin2_name'], fontsize=8.5, color='firebrick', fontweight='bold'))
for _, r in top5_over.iterrows():
    texts.append(ax.text(r['log_deaths'], r['log_aid'], r['admin2_name'], fontsize=8.5, color='darkorange', fontweight='bold'))

adjust_text(
    texts, ax=ax,
    arrowprops=dict(arrowstyle='-', color='gray', lw=0.8, alpha=0.7),
    expand_points=(3.0, 4.0),
    expand_text=(2.5, 3.5),
    force_text=(1.5, 2.0),
    force_points=(1.5, 2.0),
    lim=1000,
    only_move={'points': 'y', 'texts': 'xy', 'objects': 'xy'},
)

ax.set_xlabel('Log(1 + Avg Deaths per Month)', fontsize=11)
ax.set_ylabel('Log Avg Aid Spend per Month (USD)',              fontsize=11)
ax.set_title('Average Violence vs. Aid Spending by Town/City per Month (2021–2026)', fontsize=13)
ax.grid(True, linestyle='--', alpha=0.3, zorder=1)

underserved = town[town['category'] == 'underserved']
overserved  = town[town['category'] == 'overserved']
legend_elements = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor='firebrick',  markersize=8,
           label=f'Underserved (bottom-quartile log($/death), n={len(underserved)})'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='darkorange', markersize=8,
           label=f'Overserved  (top-quartile log($/death) + top-quartile aid, n={len(overserved)})'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='steelblue',  markersize=8,
           label=f'Other towns (n={len(town[town["category"]=="other"])})'),
]
ax.legend(handles=legend_elements, loc='upper center',
          bbox_to_anchor=(0.5, -0.10), ncol=2, fontsize=8.5, framealpha=0.85)

plt.tight_layout()
plt.savefig(OUT_DIR / 'scatter_conflict_aid.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.close()
print(f'Saved → {OUT_DIR / "scatter_conflict_aid.png"}')


Looks like you are using a tranform that doesn't support FancyArrowPatch, using ax.annotate instead. The arrows might strike through texts. Increasing shrinkA in arrowprops might help.


Loaded 11,808 rows | 164 towns | 72 months

Category counts:
category
other          94
underserved    41
overserved     29

Metric thresholds — underserved ≤ 9.86 | overserved ≥ 14.34 (metric) & ≥ 15.22 (log aid)


Saved → /Users/jackzipper/QSS20/final_project/output/scatter_conflict_aid.png
